# Tensor Parallelism Tutorial

## Overview

Tensor parallelism splits individual layers across multiple GPUs, enabling training of models with layers too large for single GPU memory.

### Learning Objectives
- Understand column and row parallel linear layers
- Implement tensor parallel attention and MLP
- Analyze communication patterns

### References
- Shoeybi et al., "Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism", arXiv 2019

## 1. Mathematical Foundation

### 1.1 Column Parallel Linear

For $Y = XW$ where $W \in \mathbb{R}^{d \times k}$, split $W$ column-wise across $N$ GPUs:

$$W = [W_1 | W_2 | ... | W_N]$$

Each GPU $i$ computes: $Y_i = XW_i$

Result: $Y = [Y_1 | Y_2 | ... | Y_N]$ (concatenated)

### 1.2 Row Parallel Linear

Split $W$ row-wise and $X$ column-wise:

$$W = \begin{bmatrix} W_1 \\ W_2 \\ ... \\ W_N \end{bmatrix}, \quad X = [X_1 | X_2 | ... | X_N]$$

Each GPU computes: $Y_i = X_i W_i$

Result: $Y = \sum_{i=1}^{N} Y_i$ (AllReduce)

### 1.3 Communication Complexity

Per transformer layer with tensor parallelism:
- Forward: 2 AllReduce operations
- Backward: 2 AllReduce operations
- Total: $O(4 \cdot B \cdot S \cdot H)$ per layer

## 2. Architecture Visualization

```
Column Parallel (MLP first layer):
┌─────────────────────────────────────────┐
│  Input X [B, S, H]                      │
│       │                                 │
│       ▼ (broadcast)                     │
│  ┌────┴────┬────┴────┐                  │
│  │ GPU 0   │ GPU 1   │                  │
│  │ W[:,:k] │ W[:,k:] │                  │
│  │ Y0      │ Y1      │                  │
│  └────┬────┴────┬────┘                  │
│       │         │                       │
│       ▼ (no comm needed)                │
│  Output [Y0 | Y1]                       │
└─────────────────────────────────────────┘

Row Parallel (MLP second layer):
┌─────────────────────────────────────────┐
│  Input [Y0 | Y1]                        │
│  ┌────┴────┬────┴────┐                  │
│  │ GPU 0   │ GPU 1   │                  │
│  │ W[0:k,:]│ W[k:,:] │                  │
│  │ Z0      │ Z1      │                  │
│  └────┬────┴────┬────┘                  │
│       │         │                       │
│       ▼ AllReduce                       │
│  Output Z = Z0 + Z1                     │
└─────────────────────────────────────────┘
```

In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
from typing import Optional

print(f"PyTorch version: {torch.__version__}")

## 3. Column Parallel Linear Implementation

In [ ]:
class ColumnParallelLinear(nn.Module):
    """Linear layer with column-wise weight partitioning.
    
    Splits output features across GPUs. Each GPU stores W[:, start:end].
    """
    
    def __init__(self, in_features: int, out_features: int, 
                 world_size: int, rank: int, bias: bool = True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.world_size = world_size
        self.rank = rank
        
        # Each GPU gets out_features / world_size columns
        assert out_features % world_size == 0
        self.local_out_features = out_features // world_size
        
        self.weight = nn.Parameter(
            torch.empty(self.local_out_features, in_features)
        )
        self.bias = nn.Parameter(
            torch.empty(self.local_out_features)
        ) if bias else None
        
        self._init_weights()
    
    def _init_weights(self):
        nn.init.kaiming_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq, in_features]
        # output: [batch, seq, local_out_features]
        return nn.functional.linear(x, self.weight, self.bias)

## 4. Row Parallel Linear Implementation

In [ ]:
class RowParallelLinear(nn.Module):
    """Linear layer with row-wise weight partitioning.
    
    Splits input features across GPUs. Requires AllReduce after forward.
    """
    
    def __init__(self, in_features: int, out_features: int,
                 world_size: int, rank: int, bias: bool = True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.world_size = world_size
        self.rank = rank
        
        # Each GPU gets in_features / world_size rows
        assert in_features % world_size == 0
        self.local_in_features = in_features // world_size
        
        self.weight = nn.Parameter(
            torch.empty(out_features, self.local_in_features)
        )
        # Bias only on rank 0 to avoid duplication after AllReduce
        self.bias = nn.Parameter(
            torch.empty(out_features)
        ) if bias and rank == 0 else None
        
        self._init_weights()
    
    def _init_weights(self):
        nn.init.kaiming_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq, local_in_features]
        output = nn.functional.linear(x, self.weight)
        
        # AllReduce to sum partial results
        if dist.is_initialized():
            dist.all_reduce(output)
        
        if self.bias is not None:
            output = output + self.bias
        
        return output

## 5. Tensor Parallel MLP

In [ ]:
class TensorParallelMLP(nn.Module):
    """MLP with tensor parallelism.
    
    Architecture:
        Input -> ColumnParallel -> GELU -> RowParallel -> Output
    
    Communication: 1 AllReduce in forward, 1 in backward
    """
    
    def __init__(self, hidden_size: int, intermediate_size: int,
                 world_size: int, rank: int):
        super().__init__()
        
        # First linear: column parallel (no comm needed)
        self.fc1 = ColumnParallelLinear(
            hidden_size, intermediate_size, world_size, rank
        )
        
        self.activation = nn.GELU()
        
        # Second linear: row parallel (AllReduce after)
        self.fc2 = RowParallelLinear(
            intermediate_size, hidden_size, world_size, rank
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)           # Column parallel, no comm
        x = self.activation(x)     # Local computation
        x = self.fc2(x)           # Row parallel, AllReduce
        return x

## 6. Tensor Parallel Attention

In [ ]:
class TensorParallelAttention(nn.Module):
    """Multi-head attention with tensor parallelism.
    
    Splits attention heads across GPUs.
    Each GPU handles num_heads / world_size heads.
    """
    
    def __init__(self, hidden_size: int, num_heads: int,
                 world_size: int, rank: int):
        super().__init__()
        assert num_heads % world_size == 0
        
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.local_num_heads = num_heads // world_size
        self.head_dim = hidden_size // num_heads
        
        # QKV projection: column parallel
        self.qkv = ColumnParallelLinear(
            hidden_size, 3 * hidden_size, world_size, rank, bias=False
        )
        
        # Output projection: row parallel
        self.out_proj = RowParallelLinear(
            hidden_size, hidden_size, world_size, rank
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape
        
        # QKV projection (column parallel)
        qkv = self.qkv(x)  # [B, S, 3 * local_hidden]
        qkv = qkv.reshape(B, S, 3, self.local_num_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        
        # Attention computation (local)
        q = q.transpose(1, 2)  # [B, local_heads, S, head_dim]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        attn = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        
        # Reshape and project (row parallel with AllReduce)
        out = out.transpose(1, 2).reshape(B, S, -1)
        out = self.out_proj(out)
        
        return out

## 7. Summary

### Key Takeaways

1. **Column Parallel**: Split output dimension, no forward comm
2. **Row Parallel**: Split input dimension, requires AllReduce
3. **Attention**: Split heads across GPUs
4. **MLP**: Column → Row pattern minimizes communication

### Communication Cost

| Operation | Forward | Backward |
|-----------|---------|----------|
| Column Parallel | 0 | AllReduce |
| Row Parallel | AllReduce | 0 |
| Per Layer | 2 AllReduce | 2 AllReduce |